# kbprojection: every class and model called once

Each code cell below focuses on exactly one class/model from the repository and prints a small output from constructing or calling it. Abstract base classes are demonstrated by attempting construction and printing the expected error. Provider/API-backed classes are constructed without making live network calls.

Use a Python 3.10+ kernel. The repository currently uses `str | dict` type hints, which fail under Python 3.9.


In [1]:
# Setup: import the local repository and shared demo objects.
import sys
import warnings
from pathlib import Path
from pprint import pprint

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))

print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())

import kbprojection
print("kbprojection loaded from:", kbprojection.__file__)

from kbprojection.models import (
    NLILabel,
    NLIProblem,
    LLMKBInjection,
    LLMKBResponse,
    LangProResult,
    ExperimentStepStatus,
    ExperimentStatus,
    ExperimentResult,
    TestMode,
    ProblemConfig,
    KBResult,
)
from kbprojection.data import KB_injection, KB_injections
from kbprojection.loaders.base import DatasetLoader
from kbprojection.loaders.snli import SNLILoader
from kbprojection.loaders.sick import SICKLoader
from kbprojection.llm import GenericAIClient
from kbprojection.langpro import (
    PrologTerm,
    CCGCat,
    CaTy,
    TreeLike,
    Atomic,
    Atom,
    Integer,
    Float,
    Var,
    Compound,
    AtomCaTy,
    CompCaTy,
    TreeLeaf,
    TT,
    AppTT,
    AbsTT,
    TLP,
    TreeNode,
    RuleApp,
)

def compact_model(obj):
    if hasattr(obj, "model_dump"):
        pprint(obj.model_dump())
    else:
        print(repr(obj))

DEMO_PROBLEM = NLIProblem(
    id="demo-1",
    premises=["A dog runs in a park."],
    hypothesis="An animal moves outdoors.",
    gold_label=NLILabel.ENTAILMENT,
    dataset="manual",
    split="demo",
)

TT_DOG = {"functor": ",", "args": [{"functor": "tlp", "args": ["dogs", "dog", "NNS"]}, "N"]}
TT_ANIMAL = {"functor": ",", "args": [{"functor": "tlp", "args": ["animals", "animal", "NNS"]}, "N"]}
TT_VAR = {"functor": ",", "args": ["X", "e"]}

print("Setup complete.")


Python: 3.11.13
Working directory: /Users/jorrytdejong/Documents/RNL paper publication
kbprojection loaded from: /Users/jorrytdejong/Documents/RNL paper publication/kbprojection/__init__.py
Setup complete.


## Pydantic models and enums


In [2]:
# Class/model: NLILabel
label = NLILabel("entailment")
print("Constructed:", label)
print("Value:", label.value)
print("All labels:", [item.value for item in NLILabel])


Constructed: NLILabel.ENTAILMENT
Value: entailment
All labels: ['entailment', 'contradiction', 'neutral', '-']


In [3]:
# Class/model: NLIProblem
obj = NLIProblem(
    id="manual-001",
    premises=["A little girl runs down the street."],
    hypothesis="A human is running outdoors.",
    gold_label=NLILabel.ENTAILMENT,
    dataset="manual",
    split="demo",
)
compact_model(obj)


{'dataset': 'manual',
 'gold_label': <NLILabel.ENTAILMENT: 'entailment'>,
 'hypothesis': 'A human is running outdoors.',
 'id': 'manual-001',
 'original_data': None,
 'premises': ['A little girl runs down the street.'],
 'split': 'demo'}


In [4]:
# Class/model: LLMKBInjection
obj = LLMKBInjection(KB_injection="isa_wn(girl, human)")
compact_model(obj)


{'KB_injection': 'isa_wn(girl, human)'}


In [5]:
# Class/model: LLMKBResponse
obj = LLMKBResponse(
    output=[
        LLMKBInjection(KB_injection="isa_wn(girl, human)"),
        LLMKBInjection(KB_injection="isa_wn(street, outdoors)"),
    ]
)
compact_model(obj)


{'output': [{'KB_injection': 'isa_wn(girl, human)'},
            {'KB_injection': 'isa_wn(street, outdoors)'}]}


In [6]:
# Class/model: LangProResult
obj = LangProResult(
    label=NLILabel.NEUTRAL,
    kb=["isa_wn(dog, animal)"],
    proofs={"entailment": "open", "contradiction": "open"},
)
compact_model(obj)


{'ccg_terms': [],
 'ccg_trees': [],
 'error': None,
 'kb': ['isa_wn(dog, animal)'],
 'label': <NLILabel.NEUTRAL: 'neutral'>,
 'llfs': [],
 'proofs': {'contradiction': 'open', 'entailment': 'open'},
 'terms': []}


In [4]:
# Class/model: ExperimentStepStatus
status = ExperimentStepStatus("success")
print("Constructed:", status)
print("Value:", status.value)
print("All statuses:", [item.value for item in ExperimentStepStatus])


Constructed: ExperimentStepStatus.SUCCESS
Value: success
All statuses: ['pending', 'success', 'failure', 'skipped', 'error']


In [5]:
# Class/model: ExperimentStatus
status = ExperimentStatus("fixed")
print("Constructed:", status)
print("Value:", status.value)
print("All statuses:", [item.value for item in ExperimentStatus])


Constructed: ExperimentStatus.FIXED
Value: fixed
All statuses: ['unknown', 'already_correct', 'fixed', 'fixed_raw_kb', 'still_wrong', 'still_wrong_raw_kb', 'error_no_kb', 'error_with_kb', 'llm_error', 'empty_kb_after_filter']


In [6]:
# Class/model: ExperimentResult
obj = ExperimentResult(
    problem=DEMO_PROBLEM,
    pred_no_kb=NLILabel.NEUTRAL,
    kb_raw=["isa_wn(dog, animal)"],
    kb_filtered=["isa_wn(dog, animal)"],
    final_status=ExperimentStatus.FIXED,
    fixed_by="filtered_kb",
)
compact_model(obj)


{'ablation_results': None,
 'ablation_subsets': None,
 'essential_kb': None,
 'final_status': <ExperimentStatus.FIXED: 'fixed'>,
 'fixed_by': 'filtered_kb',
 'kb_details': None,
 'kb_filtered': ['isa_wn(dog, animal)'],
 'kb_raw': ['isa_wn(dog, animal)'],
 'pred_no_kb': <NLILabel.NEUTRAL: 'neutral'>,
 'pred_with_kb': None,
 'pred_with_raw_kb': None,
 'problem': {'dataset': 'manual',
             'gold_label': <NLILabel.ENTAILMENT: 'entailment'>,
             'hypothesis': 'An animal moves outdoors.',
             'id': 'demo-1',
             'original_data': None,
             'premises': ['A dog runs in a park.'],
             'split': 'demo'},
 'prover_calls': None,
 'status_no_kb': <ExperimentStepStatus.PENDING: 'pending'>,
 'status_with_kb': <ExperimentStepStatus.PENDING: 'pending'>,
 'status_with_raw_kb': <ExperimentStepStatus.PENDING: 'pending'>}


In [8]:
# Class/model: TestMode
mode = TestMode("both")
print("Constructed:", mode)
print("Value:", mode.value)
print("All modes:", [item.value for item in TestMode])


Constructed: TestMode.BOTH
Value: both
All modes: ['no_kb', 'raw_kb', 'filtered', 'both', 'full']


In [11]:
# Class/model: ProblemConfig
obj = ProblemConfig(
    llm_provider="openai",
    model="gpt-4o",
    prompt_style="icl",
    test_mode=TestMode.BOTH,
    run_ablation=True,
    verbose=False,
)
compact_model(obj)


{'llm_provider': 'openai',
 'model': 'gpt-4o',
 'post_process': True,
 'prompt_style': 'icl',
 'run_ablation': True,
 'test_mode': <TestMode.BOTH: 'both'>,
 'verbose': False}


In [9]:
# Class/model: KBResult
obj = KBResult(
    relation="isa_wn(dog, animal)",
    provenance="llm",
    original_text="isa_wn(dog, animal)",
)
compact_model(obj)
print("str(obj):", str(obj))


{'original_text': 'isa_wn(dog, animal)',
 'provenance': 'llm',
 'relation': 'isa_wn(dog, animal)'}
str(obj): isa_wn(dog, animal)


## Legacy data models


In [10]:
# Class/model: KB_injection
obj = KB_injection(KB_injection="disj(sit, stand)")
compact_model(obj)


{'KB_injection': 'disj(sit, stand)'}


In [11]:
# Class/model: KB_injections
obj = KB_injections(
    output=[
        KB_injection(KB_injection="disj(sit, stand)"),
        KB_injection(KB_injection="isa_wn(dog, animal)"),
    ]
)
compact_model(obj)


{'output': [{'KB_injection': 'disj(sit, stand)'},
            {'KB_injection': 'isa_wn(dog, animal)'}]}


## Dataset loader classes


In [12]:
# Class/model: DatasetLoader
try:
    obj = DatasetLoader()
    print("Unexpectedly constructed:", obj)
except TypeError as exc:
    print("DatasetLoader is abstract, so construction gives the expected error:")
    print(type(exc).__name__ + ":", exc)


DatasetLoader is abstract, so construction gives the expected error:
TypeError: Can't instantiate abstract class DatasetLoader with abstract methods _download, _get_file_path, _get_splits, _iter_file, _parse_row


In [13]:
# Class/model: SNLILoader
obj = SNLILoader(data_dir=Path.cwd() / "demo_data" / "snli")
print("Class:", obj.__class__.__name__)
print("Data directory:", obj.data_dir)
print("Available splits:", obj._get_splits())
print("Dev file path it would use:", obj._get_file_path("dev"))


Class: SNLILoader
Data directory: /Users/jorrytdejong/Documents/RNL paper publication/demo_data/snli
Available splits: ['train', 'dev', 'test']
Dev file path it would use: /Users/jorrytdejong/Documents/RNL paper publication/demo_data/snli/snli_1.0/snli_1.0_dev.jsonl


In [17]:
# Class/model: SICKLoader
obj = SICKLoader(data_dir=Path.cwd() / "demo_data" / "sick")
print("Class:", obj.__class__.__name__)
print("Data directory:", obj.data_dir)
print("Available splits:", obj._get_splits())
print("Dev file path it would use:", obj._get_file_path("dev"))


Class: SICKLoader
Data directory: /Users/jorrytdejong/Documents/RNL paper publication/demo_data/sick
Available splits: ['train', 'dev', 'test']
Dev file path it would use: /Users/jorrytdejong/Documents/RNL paper publication/demo_data/sick/SICK_trial.txt


## LLM client class


In [18]:
# Class/model: GenericAIClient
try:
    obj = GenericAIClient(provider="openai")
    print("Constructed client wrapper.")
    print("Provider:", obj.provider)
    print("Underlying client class:", type(obj.client).__name__)
    print("No live generation call was made.")
except Exception as exc:
    print("GenericAIClient construction needs the relevant provider package/API key.")
    print(type(exc).__name__ + ":", exc)


Constructed client wrapper.
Provider: openai
Underlying client class: OpenAI
No live generation call was made.


## LangPro and Prolog-like structure classes


In [19]:
# Class/model: PrologTerm
try:
    obj = PrologTerm()
    print("Unexpectedly constructed:", obj)
except TypeError as exc:
    print("PrologTerm is abstract, so construction gives the expected error:")
    print(type(exc).__name__ + ":", exc)


PrologTerm is abstract, so construction gives the expected error:
TypeError: Can't instantiate abstract class PrologTerm with abstract methods __repr__, __str__


In [20]:
# Class/model: CCGCat
obj = CCGCat()
print("Constructed marker/base class instance.")
print("Type:", type(obj).__name__)
print("MRO:", [cls.__name__ for cls in type(obj).mro()])


Constructed marker/base class instance.
Type: CCGCat
MRO: ['CCGCat', 'ABC', 'object']


In [21]:
# Class/model: CaTy
obj = CaTy()
print("Constructed marker/base class instance.")
print("Type:", type(obj).__name__)
print("MRO:", [cls.__name__ for cls in type(obj).mro()])


Constructed marker/base class instance.
Type: CaTy
MRO: ['CaTy', 'ABC', 'object']


In [22]:
# Class/model: TreeLike
obj = TreeLike()
print("Constructed marker/base class instance.")
print("Type:", type(obj).__name__)
print("MRO:", [cls.__name__ for cls in type(obj).mro()])


Constructed marker/base class instance.
Type: TreeLike
MRO: ['TreeLike', 'ABC', 'object']


In [23]:
# Class/model: Atomic
obj = Atomic("demo_atom")
print("repr:", repr(obj))
print("str:", str(obj))
print("equals another Atomic with same value:", obj == Atomic("demo_atom"))


repr: Atomic(demo_atom)
str: demo_atom
equals another Atomic with same value: True


In [24]:
# Class/model: Atom
obj = Atom("dog")
print("repr:", repr(obj))
print("str:", str(obj))


repr: Atom(dog)
str: dog


In [25]:
# Class/model: Integer
obj = Integer(42)
print("repr:", repr(obj))
print("str:", str(obj))


repr: Integer(42)
str: 42


In [26]:
# Class/model: Float
obj = Float(3.14)
print("repr:", repr(obj))
print("str:", str(obj))


repr: Float(3.14)
str: 3.14


In [27]:
# Class/model: Var
obj = Var("X")
print("repr:", repr(obj))
print("str:", str(obj))


repr: Var(X)
str: X


In [28]:
# Class/model: Compound
obj = Compound("isa_wn", ["dog", "animal"])
print("repr:", repr(obj))
print("str:", str(obj))
print("len(str(obj)):", len(obj))


repr: Compound([isa_wn], 'dog', 'animal')
str: isa_wn(dog, animal)
len(str(obj)): 19


In [29]:
# Class/model: AtomCaTy
obj = AtomCaTy("NP:nb")
print("repr:", repr(obj))
print("str:", str(obj))
print("main:", obj.main)
print("feature:", obj.feat)


repr: AtomCaTy(NP:nb)
str: NP:nb
main: NP
feature: nb


In [30]:
# Class/model: CompCaTy
obj = CompCaTy("/", [AtomCaTy("S"), AtomCaTy("NP")])
print("repr:", repr(obj))
print("str:", str(obj))
print("functor:", obj.f)


repr: CompCaTy(AtomCaTy(S)/AtomCaTy(NP))
str: (S-NP)
functor: /


In [31]:
# Class/model: TreeLeaf
obj = TreeLeaf("t", [AtomCaTy("N"), "dogs", "dog", "NNS", "I-NP", "O"])
print("repr:", repr(obj))
print("str output:")
print(str(obj))


repr: TreeLeaf([t], AtomCaTy(N), 'dogs', 'dog', 'NNS', 'I-NP', 'O')
str output:
N
dogs
dog
NNS
I-NP
O


In [32]:
# Class/model: TT
obj = TT(TT_DOG)
print("repr:", repr(obj))
print("str:", str(obj))
print("compact:", obj.compact())
print("tree:", obj.tree())


repr: TT(TLP('dogs','dog','NNS'), AtomCaTy(N))
str: ([dogs,dog,NNS] : N)
compact: dog
tree: (N dogs,dog,NNS)


In [33]:
# Class/model: AppTT
obj = AppTT(TT_DOG, TT_ANIMAL)
print("repr:", repr(obj))
print("str:", str(obj))


repr: AppTT(TT(TLP('dogs','dog','NNS'), AtomCaTy(N)), TT(TLP('animals','animal','NNS'), AtomCaTy(N)))
str: ([dogs,dog,NNS] : N) @ ([animals,animal,NNS] : N)


In [34]:
# Class/model: AbsTT
obj = AbsTT(TT_VAR, TT_DOG)
print("repr:", repr(obj))
print("str:", str(obj))


repr: AbsTT(TT(Var(X), AtomCaTy(e)), TT(TLP('dogs','dog','NNS'), AtomCaTy(N)))
str: λ(X : e). ([dogs,dog,NNS] : N)


In [35]:
# Class/model: TLP
obj = TLP("tlp", ["dogs", "dog", "NNS"])
print("repr:", repr(obj))
print("str:", str(obj))
print("token/lemma/pos:", obj.tok, obj.lem, obj.pos)


repr: TLP('dogs','dog','NNS')
str: [dogs,dog,NNS]
token/lemma/pos: dogs dog NNS


In [36]:
# Class/model: TreeNode
raw_tree_node = {
    "functor": "trnd",
    "args": [
        {"functor": "nd", "args": [[], TT_DOG, [], "true"]},
        "node-1",
        {"functor": "axiom", "args": [[1]]},
        None,
    ],
}
obj = TreeNode(raw_tree_node)
print("str output:")
print(str(obj))
print("id:", obj.id)
print("sign:", obj.sign)
print("head compact:", obj.head.compact())


str output:
node-1:axiom([1])\ndog\nTrue
id: node-1
sign: True
head compact: dog


In [37]:
# Class/model: RuleApp
obj = RuleApp({"functor": "axiom", "args": [[1, 2]]})
print("repr:", repr(obj))
print("str:", str(obj))
print("rule:", obj.rule)
print("ids:", obj.ids)


repr: RuleApp(axiom, [1, 2], new=None, old=None)
str: axiom([1,2])
rule: axiom
ids: [1, 2]
